# ПЗ-6. Grounding и IoU

**Цель.** Оценить, насколько Qwen2.5-VL-3B-Instruct локализует объект по текстовому описанию: 200 примеров RefCOCO val, ответ — прямоугольник в JSON, метрики Acc@0.5 и средний IoU, анализ ошибок.

**Оборудование.** Google Colab, GPU T4 16 ГБ, fp16. Прогон ≈ 10–15 минут.

**Вклад.** Работа выполнена индивидуально: код, эксперименты и выводы — автор отчёта.

**Данные.** RefCOCO (val) из открытого набора `lmms-lab-encoder/RefCOCO` на Hugging Face Hub (бывший `lmms-lab/RefCOCO`, старое имя перенаправляется сюда); для анализа ошибок — открытые аннотации объектов COCO 2017. Ревизии датасета и модели зафиксированы.

In [ ]:
# Проверка устройства, на котором выполняется ноутбук
import sys, os, platform, shutil, subprocess
IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
print('Среда:', 'Google Colab' if IN_COLAB else 'локальная машина', '|', platform.platform(), '| Python', platform.python_version())
print('CPU:', platform.processor() or platform.machine(), '| логических ядер:', os.cpu_count())
try:
    import psutil
    print(f'RAM: {psutil.virtual_memory().total / 2**30:.1f} ГБ | свободно на диске: {shutil.disk_usage(".").free / 2**30:.0f} ГБ')
except ImportError:
    print(f'Свободно на диске: {shutil.disk_usage(".").free / 2**30:.0f} ГБ')
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU доступен:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не найден. В Colab: Среда выполнения → Сменить среду выполнения → GPU T4, затем перезапустите ноутбук.')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | память {props.total_memory / 2**30:.1f} ГБ | compute capability {props.major}.{props.minor} | устройств: {torch.cuda.device_count()}')
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
# свободная видеопамять и чужие процессы на GPU — через nvidia-smi (не требует создания CUDA-контекста)
q = lambda *a: subprocess.run(['nvidia-smi', *a, '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
free_mb, total_mb = map(int, q('--query-gpu=memory.free,memory.total').split(','))
print(f'Свободно видеопамяти: {free_mb / 1024:.1f} из {total_mb / 1024:.1f} ГБ')
if free_mb < 10 * 1024:
    procs = [l.split(', ') for l in q('--query-compute-apps=pid,used_memory').splitlines() if l]
    pids = [p for p, _ in procs if int(p) != os.getpid()]
    print('GPU занимают процессы (pid, МБ):', procs)
    raise RuntimeError('GPU занят другими ядрами на этом Colab-сервере. Выполните в отдельной ячейке:  !kill -9 ' + ' '.join(pids) +
                       '  — затем перезапустите ядро этого ноутбука (Restart) и Run All. '
                       'Либо Ctrl+Shift+P → Colab: Remove Server и создайте новый сервер GPU T4.')
if 'T4' not in props.name:
    print('Внимание: задание рассчитано на T4 16 ГБ (fp16). На другой GPU время и расход памяти будут отличаться.')

In [ ]:
%pip install -q transformers==4.51.3 huggingface_hub==0.30.2 accelerate==1.6.0 pyarrow

In [ ]:
import io, re, json, time, random, zipfile, platform, importlib.metadata as md
from pathlib import Path
from collections import defaultdict
from urllib.request import urlretrieve
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches, torch
from PIL import Image
import transformers
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from huggingface_hub import model_info, hf_hub_download
assert transformers.__version__ == '4.51.3', 'После %pip выполните Runtime → Restart session и запустите ячейки заново'

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Выберите в Colab среду выполнения с GPU T4'
DEVICE = 'cuda'
print('GPU:', torch.cuda.get_device_name(), '| Python', platform.python_version())
for p in ['torch', 'transformers', 'huggingface_hub', 'accelerate', 'pyarrow', 'numpy', 'pandas', 'pillow']:
    print(f'{p:16s}', md.version(p))
MID = 'Qwen/Qwen2.5-VL-3B-Instruct'; MR = '66285546d2b821cf421d4f5eb2576359d3770cd3'
DID = 'lmms-lab-encoder/RefCOCO';    DR = 'a5dff0b3194715fda69b2d6a0d2aaafb41eaa407'
print('Модель:', MID, MR, '| актуальная на Hub:', model_info(MID).sha)
print('Датасет:', DID, DR)

## 1. 200 примеров RefCOCO val

Каждая запись — изображение COCO, эталонный прямоугольник в формате COCO `[x, y, w, h]` (пиксели исходного изображения) и несколько референтных выражений; используется первое. 200 записей выбираются случайно (seed 42) из первого parquet-файла val.

In [ ]:
path = hf_hub_download(DID, 'data/val-00000-of-00004.parquet', repo_type='dataset', revision=DR)
raw = pd.read_parquet(path)
print('Записей в файле:', len(raw), '| колонки:', list(raw.columns))
idx = sorted(random.Random(SEED).sample(range(len(raw)), 200))
data = []
for i in idx:
    r = raw.iloc[i]
    img = Image.open(io.BytesIO(r['image']['bytes'])).convert('RGB')
    x, y, w, h = map(float, r['bbox'])
    data.append({'qid': r['question_id'], 'coco_id': int(r['file_name'].split('_')[2]), 'image': img,
                 'W': img.width, 'H': img.height, 'text': str(r['answer'][0]), 'gt': [x, y, x + w, y + h]})
del raw
print('Пример:', {k: v for k, v in data[0].items() if k != 'image'})
print('Уникальных изображений:', len({d['coco_id'] for d in data}))

## 2. IoU, Acc@0.5 и разбор ответа

$\mathrm{IoU}(A,B)=\dfrac{|A\cap B|}{|A\cup B|}$, прямоугольники в формате `[x1, y1, x2, y2]`. Пример считается успешным при IoU ≥ 0,5 (Acc@0.5); невалидный ответ получает IoU = 0.

In [ ]:
def area(b): return max(0., b[2] - b[0]) * max(0., b[3] - b[1])
def iou(a, b):
    inter = area([max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])])
    union = area(a) + area(b) - inter
    return inter / union if union > 0 else 0.
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1
assert iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0
assert abs(iou([0, 0, 10, 10], [5, 0, 15, 10]) - 50 / 150) < 1e-12
assert abs(iou([0, 0, 10, 10], [0, 0, 5, 5]) - .25) < 1e-12

NUM = r'(-?\d+(?:\.\d+)?)'
BOX_RE = re.compile(r'\[\s*' + r'\s*,\s*'.join([NUM] * 4) + r'\s*\]')
def parse_box(text):
    m = re.search(r'"bbox_2d"\s*:\s*' + BOX_RE.pattern, text) or BOX_RE.search(text)
    if not m: return None
    b = [float(v) for v in m.groups()]
    return b if b[2] > b[0] and b[3] > b[1] else None
assert parse_box('```json\n[{"bbox_2d": [10, 20, 110, 220], "label": "dog"}]\n```') == [10, 20, 110, 220]
assert parse_box('no box') is None and parse_box('[5, 5, 1, 1]') is None
print('Тесты IoU и разбора ответа пройдены')

## 3. Модель и запрос

Запрос просит один прямоугольник в формате, которым Qwen2.5-VL пользуется при обучении grounding: `{"bbox_2d": [x1, y1, x2, y2]}`.

In [ ]:
import gc; gc.collect()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MID, revision=MR, torch_dtype=torch.float16, attn_implementation='sdpa',
                                                           device_map=DEVICE, low_cpu_mem_usage=True).eval()   # сразу на GPU, без копии в RAM
print(f'Модель на {model.device}, занято GPU {torch.cuda.memory_allocated() / 2**30:.1f} ГБ')
model.generation_config.update(do_sample=False, temperature=None, top_p=None, top_k=None)

def make_processor(min_pixels, max_pixels):
    return AutoProcessor.from_pretrained(MID, revision=MR, min_pixels=min_pixels, max_pixels=max_pixels)

def prompt(text):
    return f'Locate the object described as "{text}" in the image. Output its bounding box in JSON format: {{"bbox_2d": [x1, y1, x2, y2]}}.'

@torch.inference_mode()
def ground(proc, img, text):
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': prompt(text)}]}]
    inputs = proc(text=[proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)], images=[img], return_tensors='pt').to(DEVICE)
    _, gh, gw = inputs['image_grid_thw'][0].tolist()
    p = proc.image_processor.patch_size              # 14: размер патча, сетка до слияния 2×2
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    answer = proc.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    return answer, gw * p, gh * p                     # ширина и высота изображения, поданного в модель

## 4. В каких координатах отвечает модель

По документации Qwen2.5-VL (технический отчёт, раздел о grounding) координаты — **абсолютные пиксели изображения после ресайза процессором** (стороны кратны 28), в отличие от Qwen2-VL, где использовалась нормировка к 0…1000. Проверяем экспериментально на 5 изображениях. Чтобы гипотезы различались, в проверке изображение принудительно уменьшается (`max_pixels = 200·28·28 ≈ 0,16 Мпикс`, ~2× по площади): при родном размере COCO ресайз почти не меняет изображение и гипотезы «исходные пиксели» и «пиксели после ресайза» неразличимы.

Гипотезы преобразования ответа `[x1, y1, x2, y2]` в координаты исходного изображения W×H (вход модели w×h):
* **resized** — `x·W/w`, `y·H/h`;
* **original** — без преобразования;
* **norm1000** — `x·W/1000`, `y·H/1000`.

In [ ]:
HYP = {
    'resized':  lambda b, W, H, w, h: [b[0] * W / w, b[1] * H / h, b[2] * W / w, b[3] * H / h],
    'original': lambda b, W, H, w, h: list(b),
    'norm1000': lambda b, W, H, w, h: [b[0] * W / 1000, b[1] * H / 1000, b[2] * W / 1000, b[3] * H / 1000],
}
check_proc = make_processor(64 * 28 * 28, 200 * 28 * 28)
check = []
for d in data[:5]:
    ans, w, h = ground(check_proc, d['image'], d['text'])
    b = parse_box(ans)
    row = {'text': d['text'], 'W×H': f"{d['W']}×{d['H']}", 'вход w×h': f'{w}×{h}', 'ответ': ans.replace('\n', ' ')[:80]}
    for k, f in HYP.items(): row[k] = iou(f(b, d['W'], d['H'], w, h), d['gt']) if b else 0.
    check.append((d, b, w, h, row))
ct = pd.DataFrame([c[-1] for c in check]); display(ct)
print('Средний IoU по гипотезам:', ct[list(HYP)].mean().round(3).to_dict())
COORD = ct[list(HYP)].mean().idxmax()
print('Выбрана система координат:', COORD)

fig, axes = plt.subplots(1, 5, figsize=(24, 5))
colors = {'resized': 'red', 'original': 'orange', 'norm1000': 'cyan'}
for ax, (d, b, w, h, row) in zip(axes, check):
    ax.imshow(d['image']); ax.axis('off'); ax.set_title(d['text'][:40], fontsize=9)
    g = d['gt']; ax.add_patch(patches.Rectangle(g[:2], g[2] - g[0], g[3] - g[1], fill=False, ec='lime', lw=3))
    if b:
        for k, f in HYP.items():
            q = f(b, d['W'], d['H'], w, h)
            ax.add_patch(patches.Rectangle(q[:2], q[2] - q[0], q[3] - q[1], fill=False, ec=colors[k], lw=2, ls='--'))
fig.legend(handles=[patches.Patch(ec='lime', fill=False, label='эталон')] + [patches.Patch(ec=c, fill=False, ls='--', label=k) for k, c in colors.items()], loc='lower center', ncol=4)
plt.show()

## 5. Прогон на 200 примерах

Основной прогон — с разрешением, близким к исходному (`min_pixels = 256·28·28`, `max_pixels = 1024·28·28`). Ответ переводится в координаты исходного изображения выбранным выше преобразованием и обрезается по границам изображения.

In [ ]:
proc = make_processor(256 * 28 * 28, 1024 * 28 * 28)
to_orig = HYP[COORD]
rows = []; t0 = time.perf_counter()
for k, d in enumerate(data):
    ans, w, h = ground(proc, d['image'], d['text'])
    b = parse_box(ans)
    pred = None
    if b:
        q = to_orig(b, d['W'], d['H'], w, h)
        pred = [min(max(q[0], 0), d['W']), min(max(q[1], 0), d['H']), min(max(q[2], 0), d['W']), min(max(q[3], 0), d['H'])]
    rows.append({'k': k, 'qid': d['qid'], 'coco_id': d['coco_id'], 'text': d['text'], 'answer': ans, 'input_wh': (w, h),
                 'gt': d['gt'], 'pred': pred, 'IoU': iou(pred, d['gt']) if pred else 0., 'valid': pred is not None})
    if (k + 1) % 50 == 0: print(f'{k + 1}/200, {time.perf_counter() - t0:.0f} с')
res = pd.DataFrame(rows)
res.to_json('pz6_predictions.json', orient='records', force_ascii=False, indent=1)
summary = pd.DataFrame([{'Примеров': len(res), 'Acc@0.5': (res.IoU >= .5).mean(), 'Средний IoU': res.IoU.mean(),
                         'Средний IoU (валидные)': res[res.valid].IoU.mean(), 'Доля валидных': res.valid.mean(),
                         'Система координат': COORD}])
display(summary.round(3))

## 6. Классификация ошибок

Для объективной разметки используются аннотации объектов COCO на тех же изображениях (RefCOCO построен на изображениях COCO train2014, которые входят в train2017):

* **невалидный ответ** — в ответе нет корректного прямоугольника `[x1, y1, x2, y2]`;
* **неверный объект** — предсказание совпадает (IoU ≥ 0,5) с *другим* размеченным объектом сильнее, чем с эталонным, либо почти не пересекается с эталоном (IoU < 0,1);
* **верный объект, неточные границы** — остальные ошибки: предсказание пересекается с эталоном (0,1 ≤ IoU < 0,5), и никакой другой объект не подходит лучше.

In [ ]:
ROOT = Path('coco'); ROOT.mkdir(exist_ok=True)
need = ['annotations/instances_train2017.json', 'annotations/instances_val2017.json']
if not all((ROOT / n).exists() for n in need):
    z = ROOT / 'annotations_trainval2017.zip'
    if not z.exists():
        print('Скачиваю аннотации COCO (~240 МБ)...')
        urlretrieve('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', z)
    with zipfile.ZipFile(z) as f:
        for n in need: f.extract(n, ROOT)
ids = set(res.coco_id); anns = defaultdict(list)
for n in need:
    with open(ROOT / n) as f: js = json.load(f)
    cat = {c['id']: c['name'] for c in js['categories']}
    for a in js['annotations']:
        if a['image_id'] in ids:
            x, y, w, h = a['bbox']; anns[a['image_id']].append({'box': [x, y, x + w, y + h], 'cat': cat[a['category_id']]})
    del js
print('Изображений с аннотациями:', sum(i in anns for i in ids), 'из', len(ids))

def classify(r):
    if not r.valid: return 'невалидный ответ'
    if r.IoU >= .5: return 'верно'
    objs = anns.get(r.coco_id, [])
    gt_obj = max(objs, key=lambda o: iou(o['box'], r.gt), default=None)          # аннотация эталонного объекта
    others = [o for o in objs if o is not gt_obj]
    best_other = max((iou(o['box'], r.pred) for o in others), default=0.)
    if (best_other >= .5 and best_other > r.IoU) or r.IoU < .1: return 'неверный объект'
    return 'верный объект, неточные границы'
res['класс'] = res.apply(classify, axis=1)
res['gt_cat'] = [max(anns.get(r.coco_id, []), key=lambda o: iou(o['box'], r.gt), default={'cat': '?'})['cat'] for r in res.itertuples()]
err = res[res['класс'] != 'верно']
cls = pd.DataFrame({'Число': res['класс'].value_counts(), 'Доля от всех': res['класс'].value_counts(normalize=True)})
cls['Доля от ошибок'] = (err['класс'].value_counts() / len(err)).reindex(cls.index)
display(cls.round(3))

res['размер'] = pd.cut([area(g) / (d['W'] * d['H']) for g, d in zip(res.gt, data)], [0, .05, .2, 1], labels=['малый (<5%)', 'средний (5–20%)', 'крупный (>20%)'])
display(res.groupby('размер', observed=True).agg(n=('IoU', 'size'), Acc05=('IoU', lambda s: (s >= .5).mean()), mIoU=('IoU', 'mean')).round(3))
print('Невалидные ответы (примеры):'); [print(' ', repr(a[:120])) for a in res[~res.valid].answer.head(5)]

## 7. Визуализация: 3 удачных и 3 неудачных примера

Зелёный — эталон, красный — предсказание модели в координатах исходного изображения.

In [ ]:
good = res[res.IoU >= .5].sort_values('IoU', ascending=False).head(3)
bad = res[(res.IoU < .5) & res.valid].sort_values('IoU').head(3)
if len(bad) < 3: bad = pd.concat([bad, res[~res.valid].head(3 - len(bad))])
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, (_, r) in zip(axes.flat, pd.concat([good, bad]).iterrows()):
    d = data[r.k]; ax.imshow(d['image']); ax.axis('off')
    g = r.gt; ax.add_patch(patches.Rectangle(g[:2], g[2] - g[0], g[3] - g[1], fill=False, ec='lime', lw=3))
    if r.pred:
        p = r.pred; ax.add_patch(patches.Rectangle(p[:2], p[2] - p[0], p[3] - p[1], fill=False, ec='red', lw=3, ls='--'))
    ax.set_title(f'«{r.text[:45]}»\nIoU={r.IoU:.2f} | {r["класс"]}', fontsize=11)
plt.tight_layout(); plt.savefig('pz6_examples.png', dpi=100); plt.show()

## 8. Выводы

In [ ]:
s = summary.iloc[0]
print(f"Acc@0.5 = {s['Acc@0.5']:.3f}, средний IoU = {s['Средний IoU']:.3f} (по валидным ответам {s['Средний IoU (валидные)']:.3f}), валидных ответов {s['Доля валидных']:.1%}.")
print(f'Система координат ответа: {COORD} (средний IoU на 5 проверочных изображениях: ' + ', '.join(f'{k} {v:.2f}' for k, v in ct[list(HYP)].mean().items()) + ').')
print('Структура ошибок (доля от всех ошибок): ' + ', '.join(f'{k} — {v:.1%}' for k, v in (err['класс'].value_counts(normalize=True)).items()) + f' (всего ошибок {len(err)} из {len(res)}).')

**Обсуждение.** Проверка на 5 изображениях с принудительным уменьшением показывает, какое преобразование координат совмещает рамки с эталоном: для Qwen2.5-VL это абсолютные пиксели входа модели после ресайза, и без пересчёта `x·W/w` рамки систематически смещаются к левому верхнему углу и уменьшаются. При неверной гипотезе (например, нормировке к 0…1000) качество падает почти до нуля, хотя модель «видит» объект правильно, — поэтому прямоугольники всегда нужно рисовать.

Ошибки типа «неверный объект» отражают ошибки понимания выражения: RefCOCO содержит много пространственных и сравнительных описаний («left guy», «second from right», «bowl behind the others»), а на изображении обычно присутствуют несколько объектов того же класса. Ошибки «неточные границы» чаще встречаются у мелких и частично перекрытых объектов, где разница в несколько пикселей заметно снижает IoU (см. таблицу по размерам объектов). Невалидные ответы возникают, когда модель отвечает текстом или возвращает несколько рамок/другой формат; их доля приведена в таблице классификации, примеры — выше.

Ограничения: 200 примеров дают стандартную ошибку Acc@0.5 около ±0,03; для каждого объекта используется только первое референтное выражение, часть которых в RefCOCO неоднозначна или содержит опечатки.

### Контрольный вопрос

**Почему Acc@0.5, а не средний IoU, стал стандартной метрикой?** Acc@0.5 отвечает на практический вопрос «найден ли нужный объект» и даёт интерпретируемую долю успехов, сопоставимую между работами (как AP@0.5 в детекции PASCAL VOC). Порог делает метрику устойчивой к неоднозначности разметки границ: разные аннотаторы рисуют рамки одного объекта с IoU порядка 0,7–0,9, поэтому различия в «хвосте» IoU мало что говорят о модели. Средний IoU смешивает два разных типа ошибок — промах мимо объекта и неточные границы правильного объекта — и сильно зависит от того, как учитываются невалидные ответы; одно и то же среднее может получиться у модели, которая всегда находит объект с IoU 0,5, и у модели, которая наполовину попадает идеально, наполовину промахивается.